# LSA - News Classification


<div dir="rtl">

# سوالات مفهومی

</dic>

## Q1

TF (Term Frequency) is basically how many times a word shows up in a single document compared to the total word count in that document. It makes sense that if a word is used a lot in an article, it must be important.

$$TF(t, d) = \frac{\text{count}(t, d)}{\sum_{t' \in d} \text{count}(t', d)}$$

IDF (Inverse Document Frequency) measures how common or rare a word is across all the documents in the dataset:

$$IDF(t, D) = \log \frac{|D|}{|\{d \in D : t \in d\}|}$$

If a word appears in almost every document, its IDF is very low because it doesn't help us tell the documents apart.

Studying them on their own is misleading because:
- **TF alone:** Really common words like 'the', 'is', and 'and' will have the highest TF scores. They appear in every document, but they don't tell us anything about the topic.
- **IDF alone:** Words that appear very rarely (like typos or rare names) will get a huge IDF score. But that doesn't mean they are important topics for the documents they are in.
- **TF-IDF:** We multiply them (TF-IDF = TF $\times$ IDF) to highlight words that are frequent in a specific document but rare across the rest of the corpus. This gives us the actual topic keywords.

## Q2


When reducing the rank in SVD, we want to keep the singular values that capture actual information (latent concepts) and discard the tail that represents noise. Plotting the singular values in decreasing order (the scree plot) typically shows a steep drop followed by a long, flat tail.

The elbow point is where the curve bends from steep to flat. Past this point, adding more components gives us diminishing returns (we are just capturing noise). A reproducible way to find it programmatically (instead of just guessing) is to draw a straight line connecting the first and last point of the curve, calculate the perpendicular distance from each point on the curve to that line, and pick the point with the maximum distance as the elbow.

## Q3

The reconstruction error tells us how much information we lose when we approximate our standardized matrix $A$ with a lower-rank matrix $A_k = U_k S_k V_k^T$.

Mathematically, it is the Frobenius norm of the difference:

$$\text{Error} = \Vert{}A - A_k\Vert{}_F = \sqrt{\sum_{i,j} (A_{ij} - (A_k)_{ij})^2}$$

We don't actually need to reconstruct the matrix to find this error. Because of SVD properties (Eckart-Young theorem), the error is just the square root of the sum of the squares of the singular values we threw away:

$$\Vert{}A - A_k\Vert{}_F = \sqrt{\sigma_{k+1}^2 + \sigma_{k+2}^2 + \dots + \sigma_r^2}$$

To make it easier to compare across different datasets, we divide this by the Frobenius norm of the original matrix to get the relative reconstruction error:

$$\text{Error}_{\text{rel}} = \frac{\Vert{}A - A_k\Vert{}_F}{\Vert{}A\Vert{}_F}$$

## Q4

Cosine similarity measures the cosine of the angle between two vectors:

$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \, \|\mathbf{v}\|}$$

- Range: $[-1, 1]$ (or $[0, 1]$ for non-negative values like word counts).
- $1$ means they point in the exact same direction (highly related concepts), $0$ means they are orthogonal (uncorrelated), and $-1$ means they point in opposite directions.
- The best part is that it is scale-invariant—it only cares about the direction, so a short article and a long article on the same topic will still have high similarity.

Euclidean distance measures the straight-line distance between two points:

$$d(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\|_2 = \sqrt{\sum_i (u_i - v_i)^2}$$

- Range: $[0, \infty)$.
- $0$ means the vectors are identical.
- The downside is that it is very sensitive to vector magnitude. If one document is much longer than another, their Euclidean distance will be huge, even if they discuss the same topics.

## Q5


Standardization scales each word feature across all documents so they have a mean of 0 and a variance of 1:

$$x'_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

SVD looks for directions of maximum variance in the data. If we don't standardize, words that are naturally very common (like 'said' or 'new') will have huge raw variances just because of their high counts. SVD would align its components along these common words instead of finding the actual hidden concepts.
Standardizing puts every word on equal footing, so SVD can find concepts based on how words co-occur rather than just how common they are.

## Q6


For a large matrix $A \in \mathbb{R}^{m \times n}$, computing the exact SVD is very slow and takes $\mathcal{O}(mn\min(m,n))$ time. If we have a massive dataset, this is too slow.
Randomized SVD solves this by projecting the large matrix onto a much smaller random subspace, doing the exact SVD on that small matrix, and then projecting the results back. This is way faster and only costs a tiny loss in accuracy.

**Pseudocode:**
1. Draw a random Gaussian matrix $\Omega \in \mathbb{R}^{n \times k}$ (where $k$ is the target rank).
2. Project the matrix: $Y = A\Omega$.
3. (Optional, improves accuracy) Run $q$ power iterations: $Y = A(A^T Y)$.
4. Find an orthonormal basis of $Y$ using QR decomposition: $Q, R = \text{qr}(Y)$.
5. Project $A$ onto this basis: $B = Q^T A$ (shape $k \times n$).
6. Compute the exact SVD of the small matrix $B$: $U_B, S, V^T = \text{svd}(B)$.
7. Map the left singular vectors back: $U = Q U_B$.
8. Return $U, S, V^T$.

In [14]:
import pandas as pd
df = pd.read_csv("dataset/df_file.csv")
print("Dataset loaded successfully, shape:", df.shape)


Dataset loaded successfully, shape: (2225, 2)


## Q7. Preprocessing

In [16]:
import re

def cleanse_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

df['Text'] = df['Text'].apply(cleanse_text)

## Visualization

### Table Summary

In [19]:
df.head()

### Labels Frequency

In [21]:
import matplotlib.pyplot as plt

label_frequency = df['Label'].value_counts()

plt.figure(figsize=(8, 4))
label_frequency.sort_index().plot(kind='bar')
plt.xlabel('Label')
plt.ylabel('Frequency')
plt.title('Frequency of Each Label')
plt.xticks(rotation=0)
plt.show()


<string>:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Q8.

In [23]:
from collections import Counter
import string


all_text = " ".join(df['Text'])

all_text = all_text.lower()
all_text = re.sub(f"[{string.punctuation}]", " ", all_text)
words = all_text.split()

word_counts = Counter(words)

most_common_words = word_counts.most_common(30)

words, counts = zip(*most_common_words)

plt.figure(figsize=(10, 6))
plt.bar(words, counts)
plt.xlabel('Words')
plt.ylabel('Frequency')
plt.title('Most Frequent Words in All Documents')
plt.xticks(rotation=45)
plt.show()


<string>:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


> The 30 most frequent words are almost all generic terms (like 'said', 'year', 'also', 'mr') that show up in pretty much any news article regardless of topic.
Raw frequency alone doesn't really tell us what a document is about. This is exactly the TF vs. TF-IDF problem from Q1: to find actual topic signal, we need words that are frequent in a specific document but not frequent everywhere. This is why the curated vocabulary in 'words.csv' is much more useful than raw word counts.

### Word Cloud

In [26]:
!pip install wordcloud

## Q9. Word Cloud

In [28]:
from wordcloud import WordCloud


def generate_word_cloud(text, title):
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text)
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(title)
    plt.axis('off')
    plt.show()

text = " ".join(df['Text'])
generate_word_cloud(text, f'Word Cloud for whole Texts')

<string>:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


> **Word Cloud Interpretation:**
> The font size of each word in the cloud is proportional to its frequency or importance score in the text. Stop words are usually removed beforehand. The colors and layouts are just for styling and don't carry any mathematical meaning. Word clouds are a nice way to get a quick visual summary of the main themes in a document collection.

In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
documents = df['Text'].str.lower().str.replace(f"[{string.punctuation}]", " ", regex=True)

vectorizer = TfidfVectorizer(stop_words='english', max_features=150)
X = vectorizer.fit_transform(documents)

feature_names = vectorizer.get_feature_names_out()

feature_names

In [31]:
important_words = pd.read_csv("words.csv").squeeze().tolist()

## Q10

## Bag of Words

In [34]:
def count_words(text):
    word_count = {word: 0 for word in important_words}
    for word in text.split():
        if word in word_count:
            word_count[word] += 1
    return word_count

df['Word_Counts'] = df['Text'].apply(count_words)
word_counts_df_all = pd.DataFrame(df['Word_Counts'].tolist(), index=df.index)

df_train = df.iloc[:2000].reset_index(drop=True)
df_test = df.iloc[2000:].reset_index(drop=True)

word_counts_df = word_counts_df_all.iloc[:2000].reset_index(drop=True)
word_counts_df_test = word_counts_df_all.iloc[2000:].reset_index(drop=True)

print('Training BoW matrix shape:', word_counts_df.shape)
print('Test BoW matrix shape:', word_counts_df_test.shape)
word_counts_df

Training BoW matrix shape: (2000, 52)
Test BoW matrix shape: (225, 52)


In [35]:
word_counts_df.describe()

## Standardization

In [37]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
word_counts_scaled = scaler.fit_transform(word_counts_df)

## Q11

In [39]:
import numpy as np

U_full, S_full, Vt_full = np.linalg.svd(word_counts_scaled, full_matrices=False)

print('Full SVD shapes (Q11):')
print(' U shape :', U_full.shape)
print(' S shape :', S_full.shape, '(diagonal of Sigma, stored as a vector)')
print(' Vt shape:', Vt_full.shape)

Full SVD shapes (Q11):
 U shape : (2000, 52)
 S shape : (52,) (diagonal of Sigma, stored as a vector)
 Vt shape: (52, 52)


## Q12

In [41]:
def find_elbow(values):
    n = len(values)
    x = np.arange(1, n + 1, dtype=float)
    y = np.asarray(values, dtype=float)
    p1 = np.array([x[0], y[0]])
    p2 = np.array([x[-1], y[-1]])
    line_vec = (p2 - p1) / np.linalg.norm(p2 - p1)
    distances = []
    for xi, yi in zip(x, y):
        p = np.array([xi, yi])
        proj = p1 + np.dot(p - p1, line_vec) * line_vec
        distances.append(np.linalg.norm(p - proj))
    return int(np.argmax(distances)) + 1

In [42]:
total_variance = np.sum(S_full**2)
explained_variance_ratio = (S_full**2) / total_variance
cumulative_variance_ratio = np.cumsum(explained_variance_ratio)

n_components = find_elbow(S_full)
print(f'Elbow point: k = {n_components}')

Elbow point: k = 11


In [43]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, len(S_full) + 1), S_full, 'o-', color='purple', markersize=4)
plt.axvline(x=n_components, color='red', linestyle='--', label=f'Elbow Point (k={n_components})')
plt.xlabel('Component Index')
plt.ylabel('Singular Value')
plt.title('Scree Plot of Singular Values')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumulative_variance_ratio) + 1), cumulative_variance_ratio, 'o-', color='blue', markersize=4)
plt.axvline(x=n_components, color='red', linestyle='--', label=f'Elbow Point (k={n_components})')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance Ratio')
plt.title('Cumulative Explained Variance Ratio Plot')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

<string>:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


> The singular values decay gradually instead of showing a single sharp knee. This happens because we are working with 52 fairly independent words spread across 5 topics, so they don't collapse onto just a few dominant directions. However, using the distance-to-chord rule still gives us a principled, reproducible way to choose $k = 7$ (or $k=11$ depending on the spectrum) instead of just picking a number randomly.

In [45]:
U_k = U_full[:, :n_components]
S_k = np.diag(S_full[:n_components])
Vt_k = Vt_full[:n_components, :]

lsi_matrix = U_k @ S_k

class _TruncatedSVDResult:
    def __init__(self, components):
        self.components_ = components

svd = _TruncatedSVDResult(Vt_k)

X_reconstructed = U_k @ S_k @ Vt_k
reconstruction_error_abs = np.linalg.norm(word_counts_scaled - X_reconstructed, 'fro')
reconstruction_error_rel = reconstruction_error_abs / np.linalg.norm(word_counts_scaled, 'fro')

print('Truncated SVD details:')
print(f' Selected rank (k): {n_components}')
print(f' Absolute reconstruction error (Frobenius Norm): {reconstruction_error_abs:.4f}')
print(f' Relative reconstruction error: {reconstruction_error_rel:.4f}')


Truncated SVD details:
 Selected rank (k): 11
 Absolute reconstruction error (Frobenius Norm): 248.8014
 Relative reconstruction error: 0.7715


## Randomized SVD Implementation (Q13 & Q14)


## Q13. Implement Randomized SVD

In [48]:
def randomized_svd(A, rank, n_iter=5):
    m, n = A.shape
    omega = np.random.randn(n, rank)

    Y = A @ omega

    for _ in range(n_iter):
        Y = A @ (A.T @ Y)

    Q, _ = np.linalg.qr(Y)

    B = Q.T @ A

    Ub, S, Vt = np.linalg.svd(B, full_matrices=False)

    U = Q @ Ub

    return U, S, Vt


## Q14. Run Randomized SVD and Compare

In [50]:
U_rand, S_rand, Vt_rand = randomized_svd(word_counts_scaled, n_components)

X_reconstructed_rand = U_rand @ np.diag(S_rand) @ Vt_rand
reconstruction_error_rand_abs = np.linalg.norm(word_counts_scaled - X_reconstructed_rand, 'fro')
reconstruction_error_rand_rel = reconstruction_error_rand_abs / np.linalg.norm(word_counts_scaled, 'fro')

print(f"Randomized SVD with rank k = {n_components}:")
print(f"  Absolute reconstruction error: {reconstruction_error_rand_abs:.4f}")
print(f"  Relative reconstruction error: {reconstruction_error_rand_rel:.4f}")

print(f"\nReconstruction Error Comparison (k = {n_components}):")
print(f"  Truncated SVD:  {reconstruction_error_rel:.4f}")
print(f"  Randomized SVD: {reconstruction_error_rand_rel:.4f}")


Randomized SVD with rank k = 11:
  Absolute reconstruction error: 250.4307
  Relative reconstruction error: 0.7766

Reconstruction Error Comparison (k = 11):
  Truncated SVD:  0.7715
  Randomized SVD: 0.7766


**Q14 Comparison:**
Randomized SVD gets a reconstruction error that is extremely close to the deterministic Truncated SVD, but it runs much faster because it only performs matrix-vector products and SVD on a very small $k \times n$ matrix.
If our corpus was the size of the whole internet, exact SVD would be impossible to run due to memory and time limits. In that case, Randomized SVD is definitely the way to go because it scales roughly linearly and works great with sparse matrices, trading a tiny bit of accuracy for massive savings in compute.

## Visualize Word Embeddings

In [53]:
import seaborn as sns

def plot_heatmap(data, title , figsize , xlabel , ylabel):
    plt.figure(figsize=figsize)
    sns.heatmap(data, annot=True, cmap='viridis', fmt='.3f')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()

In [54]:
lsi_df = pd.DataFrame(lsi_matrix, columns=[f'LSI_{i+1}' for i in range(n_components)])
lsi_df['Label'] = df_train['Label']

vocab_representation = svd.components_.T

vocab_df_7 = pd.DataFrame(vocab_representation[:, :n_components], index=important_words, columns=[f'LSI_{i+1}' for i in range(n_components)])

plot_heatmap(vocab_df_7, f'Vocabulary Representation (First {n_components} Components)', (12, 16) , 'Components','Words')

<string>:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Q15

## Intuition of each component (concept)

In [57]:
top_words = {}
for i in range(n_components):
    component_loadings = vocab_representation[:, i]
    top_indices = abs(component_loadings).argsort()[-5:][::-1]
    top_words[f'Component_{i+1}'] = [(important_words[idx], component_loadings[idx]) for idx in top_indices]


for component, words in top_words.items():
    print(f"{component}:")
    for word, value in words:
        print(f"  {word}: {value:.4f}")


Component_1:
  election: 0.3332
  labour: 0.3281
  technology: -0.2570
  minister: 0.2487
  mobile: -0.2478
Component_2:
  government: 0.2701
  labour: 0.2317
  election: 0.2273
  uk: 0.2019
  work: 0.1994
Component_3:
  play: -0.3262
  time: -0.3245
  game: -0.3207
  win: -0.3106
  won: -0.2521
Component_4:
  artist: 0.3450
  music: 0.3412
  won: 0.3378
  film: 0.2977
  game: -0.2952
Component_5:
  growth: -0.2655
  economy: -0.2604
  deal: -0.2522
  phone: 0.2493
  mobile: 0.2360
Component_6:
  growth: -0.4572
  economy: -0.4017
  million: -0.2351
  expected: -0.2030
  deal: 0.1975
Component_7:
  director: 0.3459
  film: 0.3181
  home: 0.2493
  government: 0.2482
  law: 0.2476
Component_8:
  director: -0.3420
  home: 0.2919
  film: -0.2823
  firm: -0.2692
  law: 0.2472
Component_9:
  help: -0.2973
  money: -0.2832
  new: 0.2784
  work: -0.2318
  company: 0.2260
Component_10:
  club: -0.4314
  money: -0.3346
  phone: -0.3306
  mobile: -0.3185
  deal: -0.3062
Component_11:
  digital: 0

> **Component Interpretation (Q15):**
> Looking at the top 5 words with the highest magnitude, we can guess the latent concepts:
> - Component 1 is loaded on words like 'election', 'labour', 'minister', representing Politics.
> - Component 3 is loaded on words like 'play', 'game', 'win', 'won', representing Sport.
> - Component 4 is loaded on words like 'artist', 'music', 'won', 'film', representing Entertainment.
> - Component 5 & 6 are loaded on 'growth', 'economy', representing Business.
> This kind of manual interpretation is a nice sanity check, but as Q19 notes, it doesn't scale to real-world applications where we might have hundreds of components.

## Q16

In [60]:
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

def word_similarity(word1, word2, important_words):
    if word1 not in important_words or word2 not in important_words:
        return None
    word1_index = important_words.index(word1)
    word2_index = important_words.index(word2)

    word1_representation = svd.components_.T[word1_index]
    word2_representation = svd.components_.T[word2_index]

    cosine_sim = cosine_similarity([word1_representation], [word2_representation])[0][0]
    euclidean_dist = euclidean_distances([word1_representation], [word2_representation])[0][0]

    return {'cosine_similarity': cosine_sim, 'euclidean_distance': euclidean_dist}

word_pairs = [
    ('mobile', 'technology'),
    ('director', 'film'),
    ('win', 'won'),
    ('play', 'game'),
    ('play', 'law'),
    ('government', 'music')
]


for w1, w2 in word_pairs:
    sims = word_similarity(w1, w2, important_words)
    print(f'({w1}, {w2}):')
    print(f"  Cosine Similarity:  {sims['cosine_similarity']:.4f}")
    print(f"  Euclidean Distance: {sims['euclidean_distance']:.4f}")


(mobile, technology):
  Cosine Similarity:  0.5843
  Euclidean Distance: 0.4712
(director, film):
  Cosine Similarity:  0.9833
  Euclidean Distance: 0.1141
(win, won):
  Cosine Similarity:  0.7535
  Euclidean Distance: 0.3201
(play, game):
  Cosine Similarity:  0.9574
  Euclidean Distance: 0.1485
(play, law):
  Cosine Similarity:  -0.3098
  Euclidean Distance: 0.7474
(government, music):
  Cosine Similarity:  -0.0079
  Euclidean Distance: 0.7665


**Interpretation (Q16):**
Semantically related pairs that belong to the same topic (like director/film, play/game, win/won) show high cosine similarity and low Euclidean distance. Unrelated pairs (like play/law, government/music) show low similarity and larger distances. This proves that the latent space is successfully capturing topical associations.

## Q17

## Similarity between documents and words

In [64]:
import matplotlib.pyplot as plt

def document_similarity_with_words(doc_index, df, important_words):
    doc_representation = lsi_matrix[doc_index]

    word_similarities = {}
    for word in important_words:
        word_index = important_words.index(word)
        word_representation = svd.components_.T[word_index]
        similarity = cosine_similarity([doc_representation], [word_representation])[0][0]
        word_similarities[word] = similarity

    return word_similarities


STUDENT_ID_LAST3 = 318
doc_index = STUDENT_ID_LAST3 % len(lsi_matrix)
similarities = document_similarity_with_words(doc_index, lsi_df.drop(columns=['Label']), important_words)

words = list(similarities.keys())
cosine_sim_values = list(similarities.values())
word_counts = df_train['Word_Counts'].iloc[doc_index]

plt.figure(figsize=(20, 7))

plt.subplot(2, 1, 1)
plt.bar(words, cosine_sim_values, color='skyblue')
plt.xlabel('Words')
plt.ylabel('Cosine Similarity')
plt.title(f'Cosine Similarity of Document {doc_index} with Each Word in Vocabulary')
plt.xticks(rotation=80)
plt.grid(True)

plt.subplot(2, 1, 2)
plt.bar(word_counts.keys(), word_counts.values(), color='salmon')
plt.xlabel('Words')
plt.ylabel('Word Count')
plt.title(f'Bag of Words Counts for Document {doc_index}')
plt.xticks(rotation=80)
plt.grid(True)

plt.tight_layout()
plt.show()


<string>:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


**Comparison (Q17):**
The raw word-count chart is mostly zeros (very sparse), while the cosine similarity chart in the latent space is dense. Every word gets a non-zero similarity score, including words that don't literally appear in the document but are semantically related to its content.

# Querying
To find the most relevant documents to a single word.

In [67]:
from sklearn.metrics.pairwise import cosine_similarity

def find_relevant_documents(word, df, important_words):
    if word not in important_words:
        print(f"The word '{word}' is not in the vocabulary.")
        return None

    word_index = important_words.index(word)
    word_representation = svd.components_.T[word_index]

    cosine_similarities = cosine_similarity([word_representation], df)
    cosine_similarities = cosine_similarities.flatten()

    result_df = pd.DataFrame({'Document_Index': df.index, 'Cosine_Similarity': cosine_similarities})
    result_df = result_df.sort_values(by='Cosine_Similarity', ascending=False)
    relevant_indices = result_df['Document_Index'].tolist()

    return relevant_indices, cosine_similarities

word = 'game'
relevant_indices, _ = find_relevant_documents(word, lsi_df.drop(columns=['Label']), important_words)

relevant_texts = df_train.loc[relevant_indices[:10]]
relevant_texts

## Similarity between words and labels

In [69]:
word = 'mobile'
relevant_indices, cosine_similarities = find_relevant_documents(word, lsi_df.drop(columns=['Label']), important_words)

df_train['Cosine_Similarity'] = cosine_similarities

average_cosine_similarity_per_label = df_train.groupby('Label')['Cosine_Similarity'].mean()

plt.figure(figsize=(10, 6))
average_cosine_similarity_per_label.plot(kind='bar', color='teal')
plt.title(f'Average Cosine Similarity of "{word}" for Each Label')
plt.xlabel('Label')
plt.ylabel('Average Cosine Similarity')
plt.grid(True)
plt.show()


<string>:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Q18

**Discussion (Q18):**
If a document talks about phones using words like 'mobile' and 'digital' but never literally says 'technology', a raw keyword search for 'technology' will completely miss it. In the latent space, words like 'mobile', 'digital', and 'technology' end up close together because they frequently co-occur in the same contexts. Projecting the query 'technology' into the latent space will still match that document.

**Computational Cost:** A BoW search requires comparing the query against a very high-dimensional, sparse vector for each document. Latent space search compares against a dense vector of only $k$ dimensions, which is much faster and requires far less memory at scale.

## Q19

## Labels in terms of concepts

In [74]:
lsi_df['Label'] = df_train['Label']

avg_scores = lsi_df.groupby('Label').mean()

plot_heatmap(avg_scores.T, "Average Scores of LSI Components for Each Label" , (10, 8) ,"Label" ,"LSI Component")

<string>:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


**Proposed method (Q19):**
We represent each category by the average latent vector of its training documents (the heatmap above). Then, we classify a new document by projecting it into the latent space, calculating its cosine similarity to the average vector of each category, and assigning it to the category with the highest similarity.
This method is great because it is purely geometric and doesn't require us to manually interpret the meaning of each latent component, meaning it easily scales to any dataset.

## Classification

In [77]:
def label_alignment_scores(document_latent_vector, lsi_df):
    label_scores = {}
    for label in lsi_df['Label'].unique():
        label_documents = lsi_df[lsi_df['Label'] == label].drop(columns=['Label'])
        average_latent_vector = label_documents.mean(axis=0)
        similarity_score = cosine_similarity([document_latent_vector], [average_latent_vector])[0][0]
        label_scores[label] = similarity_score
    return label_scores

document_latent_vector = lsi_matrix[1280]
alignment_scores = label_alignment_scores(document_latent_vector, lsi_df)

print('Alignment Scores:')
for label, score in alignment_scores.items():
    print(f'Label {label}: {score}')


Alignment Scores:
Label 0: -0.3325238405886435
Label 1: -0.3241580365061699
Label 2: 0.8288592650509984
Label 3: -0.20518045800216467
Label 4: -0.06427760654370106


In [78]:
predicted_labels = []

for i, document_latent_vector in enumerate(lsi_matrix):
    alignment_scores = label_alignment_scores(document_latent_vector, lsi_df)
    predicted_label = max(alignment_scores, key=alignment_scores.get)
    predicted_labels.append(predicted_label)

predicted_labels = pd.Series(predicted_labels)
accuracy = (predicted_labels == lsi_df['Label']).mean()
print('Training Set Accuracy:', accuracy)


Training Set Accuracy: 0.7765


## Q20

In [80]:
word_counts_scaled_test = scaler.transform(word_counts_df_test)

lsi_matrix_test = word_counts_scaled_test @ svd.components_.T
lsi_df_test = pd.DataFrame(lsi_matrix_test, columns=[f'LSI_{i+1}' for i in range(n_components)])
lsi_df_test['Label'] = df_test['Label']

test_predicted_labels = []
for doc_vec in lsi_matrix_test:
    scores = label_alignment_scores(doc_vec, lsi_df)
    pred_label = max(scores, key=scores.get)
    test_predicted_labels.append(pred_label)

test_predicted_labels = pd.Series(test_predicted_labels)
test_accuracy = (test_predicted_labels == df_test['Label']).mean()

print('Test Accuracy:', test_accuracy)

Test Accuracy: 0.7911111111111111


In [81]:
CATEGORY_NAMES = {0: 'Politics', 1: 'Sport', 2: 'Technology', 3: 'Entertainment', 4: 'Business'}

print('\nAccuracy by category on the test set:')
for label in sorted(lsi_df['Label'].unique()):
    name = CATEGORY_NAMES.get(label, 'Unknown')
    mask = df_test['Label'] == label
    if mask.sum() > 0:
        class_acc = (test_predicted_labels[mask] == df_test['Label'][mask]).mean()
        print(f"  Category {label} ({name}): {class_acc:.4f}")
    else:
        print(f"  Category {label} ({name}): N/A (no samples in test set)")



Accuracy by category on the test set:
  Category 0 (Politics): N/A (no samples in test set)
  Category 1 (Sport): N/A (no samples in test set)
  Category 2 (Technology): N/A (no samples in test set)
  Category 3 (Entertainment): N/A (no samples in test set)
  Category 4 (Business): 0.7911


> **Test Set Split Observation:**
> The source dataset is sorted by label, so taking the literal 'last 225 rows' as the test set (as mandated by the assignment) means it only contains Category 4 (Business) documents. Because of this, we can't measure accuracy for categories 0-3 on the test set, and our overall test accuracy is just the accuracy on Business documents. This is a property of the mandated split, not a bug in the classifier.